In [ ]:
import json
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import clear_output
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve

%matplotlib inline

Обратимся к набору данных SST-2. Holdout часть данных (которая понадобится вам для посылки) доступна по ссылке ниже.

In [ ]:
!wget https://raw.githubusercontent.com/girafe-ai/ml-course/refs/heads/24f_yandex_ml_trainings/homeworks/hw04_bert_and_co/texts_holdout.json

--2026-02-18 16:41:48--  https://raw.githubusercontent.com/girafe-ai/ml-course/refs/heads/24f_yandex_ml_trainings/homeworks/hw04_bert_and_co/texts_holdout.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 51581 (50K) [text/plain]
Saving to: ‘texts_holdout.json.1’

texts_holdout.json. 100%[===================>]  50.37K  --.-KB/s    in 0.008s  

2026-02-18 16:41:48 (6.42 MB/s) - ‘texts_holdout.json.1’ saved [51581/51581]



In [ ]:
df = pd.read_csv(
    "https://github.com/clairett/pytorch-sentiment-classification/raw/master/data/SST2/train.tsv",
    delimiter="\t",
    header=None,
)
texts_train = df[0].values[:5000].tolist()
y_train = df[1].values[:5000].tolist()
texts_test = df[0].values[5000:].tolist()
y_test = df[1].values[5000:].tolist()
with open("texts_holdout.json") as iofile:
    texts_holdout = json.load(iofile)

In [ ]:
!pip install -Uqq transformers

In [ ]:
from transformers import pipeline


unmasker = pipeline('fill-mask', 'distilbert-base-uncased')
unmasker("Hello I'm a [MASK] model.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

[{'score': 0.05292870104312897,
  'token': 2535,
  'token_str': 'role',
  'sequence': "hello i ' m a role model."},
 {'score': 0.03968583047389984,
  'token': 4827,
  'token_str': 'fashion',
  'sequence': "hello i ' m a fashion model."},
 {'score': 0.03474362939596176,
  'token': 2449,
  'token_str': 'business',
  'sequence': "hello i ' m a business model."},
 {'score': 0.034622836858034134,
  'token': 2944,
  'token_str': 'model',
  'sequence': "hello i ' m a model model."},
 {'score': 0.018145203590393066,
  'token': 11643,
  'token_str': 'modeling',
  'sequence': "hello i ' m a modeling model."}]

In [ ]:
import torch
from transformers import DistilBertModel, DistilBertTokenizer, logging

logging.set_verbosity_error()
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertModel.from_pretrained('distilbert-base-uncased')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
tokenized_train_texts = tokenizer(texts_train, return_tensors='pt', padding=True)
tokenized_test_texts = tokenizer(texts_test, return_tensors='pt', padding=True)

In [ ]:
tokenized_hold_texts = tokenizer(texts_holdout, return_tensors='pt', padding=True)

In [ ]:
tokenized_train_texts['input_ids']

tensor([[  101,  1037, 18385,  ...,     0,     0,     0],
        [  101,  4593,  2128,  ...,     0,     0,     0],
        [  101,  2027,  3653,  ...,     0,     0,     0],
        ...,
        [  101,  5212,  1048,  ...,     0,     0,     0],
        [  101,  2009,  2515,  ...,     0,     0,     0],
        [  101,  1037, 20228,  ...,     0,     0,     0]])

In [ ]:
import numpy as np


device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

batch_size = 32
features_train = []
features_test = []
with torch.no_grad():
    for i in range(0, len(texts_train), batch_size):
        texts_batch = tokenized_train_texts["input_ids"][i : i + batch_size].to(device)
        masks_batch = tokenized_train_texts["attention_mask"][i : i + batch_size].to(device)
        output = model(texts_batch, masks_batch)
        batch_features = output.last_hidden_state[:, 0, :].cpu().numpy()
        features_train.append(batch_features)
    for i in range(0, len(texts_test), batch_size):
        texts_batch = tokenized_test_texts["input_ids"][i : i + batch_size].to(device)
        masks_batch = tokenized_test_texts["attention_mask"][i : i + batch_size].to(device)
        output = model(texts_batch, masks_batch)
        batch_features = output.last_hidden_state[:, 0, :].cpu().numpy()
        features_test.append(batch_features)

features_train = np.concatenate(features_train, axis=0)
features_test = np.concatenate(features_test, axis = 0)
features_train.shape

(5000, 768)

In [ ]:
import numpy as np


device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

batch_size = 32
features_hold = []
with torch.no_grad():
    for i in range(0, len(texts_holdout), batch_size):
        texts_batch = tokenized_hold_texts["input_ids"][i : i + batch_size].to(device)
        masks_batch = tokenized_hold_texts["attention_mask"][i : i + batch_size].to(device)
        output = model(texts_batch, masks_batch)
        batch_features = output.last_hidden_state[:, 0, :].cpu().numpy()
        features_hold.append(batch_features)


features_hold = np.concatenate(features_hold, axis=0)
features_hold.shape

(500, 768)

In [ ]:
import warnings
from sklearn.linear_model import LogisticRegression


warnings.simplefilter('ignore')  # Ignore warning on model fitting.
lr_clf = LogisticRegression().fit(features_train, y_train)

In [ ]:
lr_clf.score(features_test, y_test)

0.8453125

In [ ]:
y_pred_test = lr_clf.predict_proba(features_test)
y_pred_t = lr_clf.predict_proba(features_train)
y_pred_h = lr_clf.predict_proba(features_hold)